# 第四週｜F5-TTS 真實語音與多語字幕影片

180 分鐘；需 Colab T4 GPU 和 FFmpeg；**尚未完成雲端 GPU 端到端實測**。逐段執行，不可跳格；示範英文語音及人工撰寫中文字幕。

## 01 環境安裝

**逐行學習重點：**import/subprocess/check=True：安裝失敗應停止，不得假裝成功。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
import sys,subprocess,shutil
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg','fonts-noto-cjk'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','f5-tts==1.1.22','soundfile','Pillow','numpy'],check=True)
assert shutil.which('ffmpeg')
print(sys.version)

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 02 變數與工作目錄

**逐行學習重點：**Path、變數和 dict；burn_subtitles 控制是否將字幕燒錄畫面。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
from pathlib import Path
import json
from IPython.display import display,Audio,Video
ROOT=Path('/content/week4_f5_real')
INPUT=ROOT/'input';OUT=ROOT/'output'
INPUT.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
CONFIG={'fps':24,'width':1280,'height':720,'burn_subtitles':True,'speed':1.0}
print(ROOT)

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 03 參考聲音

**逐行學習重點：**if/else 分支；只有已獲授權的聲音可供條件生成。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
from importlib.resources import files
USE_UPLOAD=False
REFERENCE_TEXT='Some call me nature, others call me mother nature.'
if USE_UPLOAD:
    from google.colab import files as upload_tool
    uploaded=upload_tool.upload()
    assert uploaded,'請上傳獲授權的音檔'
    name,data=next(iter(uploaded.items()))
    ref_audio=INPUT/name;ref_audio.write_bytes(data)
    raise ValueError('請將 REFERENCE_TEXT 改成錄音逐字稿後再執行')
else:
    ref_audio=Path(str(files('f5_tts').joinpath('infer/examples/basic/basic_ref_en.wav')))
assert ref_audio.is_file(),ref_audio
print(ref_audio)

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 04 場景資料

**逐行學習重點：**List、Dictionary、for 及 json；中英內容是人工撰寫，不代表模型翻譯。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
SCENES=[
{'id':1,'text':'Welcome to our Python artificial intelligence classroom.','subtitle':'歡迎來到 Python 人工智慧教室。','color':'#12445b'},
{'id':2,'text':'Today we will generate speech with a real F five T T S model.','subtitle':'今天使用真正的 F5 TTS 模型生成語音。','color':'#285c71'},
{'id':3,'text':'Then we will export subtitles and a final video.','subtitle':'最後輸出字幕和完整影片。','color':'#236b70'}]
for scene in SCENES:
    assert scene['text'].strip() and scene['subtitle'].strip()
(INPUT/'scenes.json').write_text(json.dumps(SCENES,ensure_ascii=False,indent=2),encoding='utf-8')
print(len(SCENES))

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 05 模型載入

**逐行學習重點：**載入真實權重，可能需要網路和 GPU 記憶體。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
import torch
assert torch.cuda.is_available(),'請啟用 Colab T4 GPU'
from f5_tts.api import F5TTS
model=F5TTS(model='F5TTS_v1_Base',device='cuda')
print('F5-TTS model loaded')

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 06 真正產生 WAV

**逐行學習重點：**model.infer() 真實模型推論，輸出三段 WAV；不能用 DummyF5TTS。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
import soundfile as sf
import numpy as np
WAV_FILES=[]
for scene in SCENES:
    wav_path=OUT/f"scene_{scene['id']:02}.wav"
    wav,sr,spec=model.infer(ref_file=str(ref_audio),ref_text=REFERENCE_TEXT,gen_text=scene['text'],file_wave=str(wav_path),speed=CONFIG['speed'],seed=42+scene['id'])
    audio,sample_rate=sf.read(str(wav_path))
    assert sample_rate>0 and len(audio)>sample_rate*.3 and np.isfinite(audio).all()
    WAV_FILES.append(wav_path)
    print(wav_path.name,len(audio)/sample_rate)
assert len(WAV_FILES)==3
display(Audio(filename=str(WAV_FILES[0])))

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 07 WAV 音長與字幕

**逐行學習重點：**def/divmod/for/f-string；字幕按 WAV 實際時長生成，非 ASR 逐字對齊。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
def srt_time(seconds):
    millis=round(seconds*1000)
    h,millis=divmod(millis,3600000)
    m,millis=divmod(millis,60000)
    s,millis=divmod(millis,1000)
    return f'{h:02}:{m:02}:{s:02},{millis:03}'
TIMELINE=[];blocks=[];cursor=0.
for i,(scene,path) in enumerate(zip(SCENES,WAV_FILES),1):
    info=sf.info(str(path));duration=info.frames/info.samplerate
    start,end=cursor,cursor+duration
    blocks.append(f"{i}\n{srt_time(start)} --> {srt_time(end)}\n{scene['subtitle']}")
    TIMELINE.append({'id':scene['id'],'start':start,'end':end})
    cursor=end
SRT=OUT/'subtitles.srt'
SRT.write_text('\n\n'.join(blocks)+'\n',encoding='utf-8-sig')
(OUT/'timeline.json').write_text(json.dumps(TIMELINE,ensure_ascii=False,indent=2),encoding='utf-8')
print(SRT.read_text(encoding='utf-8-sig'))

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 08 Pillow 產生圖片

**逐行學習重點：**逐場建立 16:9 圖卡，圖片不預先繪製字幕。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
from PIL import Image,ImageDraw,ImageFont
FONT='/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
IMAGE_FILES=[]
for scene in SCENES:
    image=Image.new('RGB',(CONFIG['width'],CONFIG['height']),scene['color'])
    draw=ImageDraw.Draw(image)
    font=ImageFont.truetype(FONT,58)
    draw.text((80,100),'PYTHON × F5-TTS',font=font,fill='white')
    draw.text((80,220),f"SCENE {scene['id']:02}",font=font,fill='#f9e1bd')
    target=OUT/f"scene_{scene['id']:02}.png"
    image.save(target)
    IMAGE_FILES.append(target)
print([p.name for p in IMAGE_FILES])

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 09 FFmpeg 影片片段

**逐行學習重點：**使用真正 WAV + PNG 合成各片 MP4，輸出 H.264/AAC。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
CLIPS=[]
for scene,img,wav in zip(SCENES,IMAGE_FILES,WAV_FILES):
    clip=OUT/f"clip_{scene['id']:02}.mp4"
    cmd=['ffmpeg','-y','-loglevel','error','-loop','1','-framerate',str(CONFIG['fps']),'-i',str(img),'-i',str(wav),'-c:v','libx264','-pix_fmt','yuv420p','-c:a','aac','-shortest',str(clip)]
    subprocess.run(cmd,check=True)
    assert clip.is_file() and clip.stat().st_size>1000
    CLIPS.append(clip)
print(len(CLIPS))

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 10 合併乾淨影片

**逐行學習重點：**透過 concat 清單依序串接三個片段。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
concat_path=OUT/'concat.txt'
concat_path.write_text(''.join(f"file '{p.resolve()}'\n" for p in CLIPS),encoding='utf-8')
CLEAN=OUT/'video_no_subtitles.mp4'
subprocess.run(['ffmpeg','-y','-loglevel','error','-f','concat','-safe','0','-i',str(concat_path),'-c','copy',str(CLEAN)],check=True)
assert CLEAN.stat().st_size>1000
print(CLEAN)

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 11 字幕開關

**逐行學習重點：**if/else；關閉字幕時輸出無字幕版本，開啟則用 libass 燒錄。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
FINAL=OUT/'video.mp4'
if CONFIG['burn_subtitles']:
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',CLEAN.name,'-vf','subtitles=subtitles.srt:charenc=UTF-8','-c:v','libx264','-c:a','copy',FINAL.name],check=True,cwd=str(OUT))
else:
    shutil.copyfile(CLEAN,FINAL)
assert FINAL.stat().st_size>1000
print('字幕燒錄',CONFIG['burn_subtitles'])

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。

## 12 成果驗收 ZIP

**逐行學習重點：**實際三段 WAV、SRT、MP4 存在且影片可播放才算完成。

教師講解 → 跟做 → 執行 → 驗收。

In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
report={'engine':'F5TTS_v1_Base','scene_count':len(SCENES),'wav_count':len(WAV_FILES),'burn_subtitles':CONFIG['burn_subtitles']}
(OUT/'report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
required=[*WAV_FILES,*IMAGE_FILES,SRT,CLEAN,FINAL,OUT/'report.json']
for path in required:
    assert path.is_file() and path.stat().st_size>0,path
bundle=OUT/'student_results.zip'
with ZipFile(bundle,'w',ZIP_DEFLATED) as archive:
    for path in [*WAV_FILES,*IMAGE_FILES,SRT,FINAL,OUT/'report.json',OUT/'timeline.json']:
        archive.write(path,path.name)
print(bundle)
display(Video(filename=str(FINAL),embed=True))

### 課堂練習

先預測本節結果；修改一個合宜的輸入欄位並重跑。若報錯，記錄錯誤與原因，**不可用假波形冒充模型合成**。